# Heatmap del quality factor di un run

Per ogni punto (membrana) di un run (cartella con `griglia.csv` e un CSV per punto) fa la stessa analisi di `quality_factor.ipynb`: media mobile di CH2 (la `moving_average` di `movemin.py`, `np.convolve`), ricerca di tutte le campane **sulla curva liscia**, dentro la rampa di CH1 e senza i bordi non validi, e **Q = altezza / larghezza** di ogni campana (altezza = prominenza, larghezza = FWHM in campioni). Poi disegna la **mappa di Q con il valore scritto su ogni membrana**.

Ogni membrana ha più campane, ma la mappa ha un solo valore per membrana: lo scegli con `CRITERIO_CAMPANA` in cella 1:
- `"piu_prominente"`: la campana più alta di ogni membrana (la campana scelta può cambiare da una membrana all'altra);
- `"n"`: la n-esima campana da sinistra (`N_CAMPANA`); se una membrana ne ha meno, resta senza valore (`n.d.`);
- `"media"`: la media di Q di tutte le campane trovate.

**Due modi di disegnare la mappa** (`MAPPA`): `"reticolo"` (una cella per membrana, righe e colonne del percorso a meandro, ricostruite dalle posizioni) oppure `"posizioni"` (un quadrato per membrana alle coordinate X, Y reali, utile se il campione è inclinato o i punti non sono su una griglia). Se il reticolo non si riesce a ricostruire, passa da solo a `"posizioni"`.

I valori sono mostrati in unità di `UNITA_Q` (V per campione, di default 1e-6) per averli leggibili. Tutte le campane di ogni membrana sono salvate in `heatmap_q_campane.csv`, e la Q scelta in `heatmap_q_valori.csv`, nella cartella del run.

Per decidere se una membrana ha davvero una risonanza: una campana conta nella mappa solo se la sua altezza è almeno `SOGLIA_RUN` (10%) di quella della campana più alta trovata in **tutto il run** (e almeno `ALTEZZA_MIN_V` se la imposti). Così una membrana senza risonanze, dove ci sono solo gobbi di rumore, resta `n.d.`. Tutte le campane trovate sono salvate in `heatmap_q_campane.csv`: cambiare criterio, `SOGLIA_RUN` o `ALTEZZA_MIN_V` non richiede di rileggere i CSV, ma cambiare i parametri di analisi (`WINDOW_LENGTH`, `SOGLIA_PROMINENZA`...) sì (`RICALCOLA = True`).

Ordine: cella 1 (scrivi `CARTELLA_RUN` e scegli il criterio), 2 (registro), 3 (calcolo, può durare qualche minuto), 4 (controllo su una membrana), 5 (mappa).

In [ ]:
# Cella 1 - impostazioni e funzioni
import csv
import os
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from scipy.signal import find_peaks, peak_widths

try:
    import pandas as pd
except ImportError:
    pd = None

CARTELLA_RUN = ""              # <-- SCRIVI QUI la cartella del run, es. "2026-10-06/run_007" (relativa al notebook) o un percorso completo

CRITERIO_CAMPANA = "piu_prominente"   # "piu_prominente" | "n" | "media"
N_CAMPANA = 1                  # per CRITERIO_CAMPANA = "n": quale campana da sinistra (1 = la prima)
MAPPA = "reticolo"             # "reticolo" | "posizioni"
UNITA_Q = 1e-6                 # Q mostrata in unita' di questo valore (V per campione)
FORMATO_NUMERO = "{:.2f}"      # come scrivere il numero sulla membrana
VMIN = None                    # estremi della scala colori; None = automatici
VMAX = None

CANALE = "CH2"
CANALE_RAMPA = "CH1"           # limita la ricerca alla rampa di questa colonna; None = tutto il record
CAMPANE_VERSO_ALTO = True      # True: risonanze = massimi; False: minimi
WINDOW_LENGTH = 10000          # lunghezza della media mobile in campioni (come window_length di movemin.py)
SOGLIA_PROMINENZA = 0.1        # una campana conta se la sua altezza e' almeno questa frazione dell'escursione max-min nella zona
SNR_MIN = 10                   # ... e almeno questo multiplo del rumore residuo della curva liscia (taglia i gobbi piu' piccoli del rumore)
SOGLIA_RUN = 0.1               # per la mappa una campana conta se e' alta almeno questa frazione della campana piu' alta di TUTTO il run (le membrane senza risonanze restano n.d.)
ALTEZZA_MIN_V = None           # altezza minima assoluta (V) per la mappa; None = nessuna oltre a SOGLIA_RUN
DISTANZA_MIN = None            # distanza minima tra campane in campioni; None = WINDOW_LENGTH
FRAZIONE_ALTEZZA = 0.5         # a che frazione dell'altezza si misura la larghezza (0.5 = FWHM)
V_INIZIO_V = 0.9               # rampa di CH1: inizio quando CH1 supera questo valore
TOLLERANZA_FINE_V = 0.3        # rampa di CH1: fine quando CH1 e' a meno di questo dal suo massimo

RICALCOLA = False              # True: ignora i file salvati e rilegge tutti i CSV (da usare dopo aver cambiato i parametri di analisi)
PUNTO_CONTROLLO = 0            # indice del punto mostrato nella cella 4


def moving_average(values, window_length):
    """Media mobile con finestra di lunghezza variabile."""   # identica a movemin.py
    if window_length <= 0:
        raise ValueError('window_length deve essere > 0')

    window_length = int(window_length)
    if window_length % 2 == 0:
        window_length += 1

    kernel = np.ones(window_length) / window_length
    return np.convolve(values, kernel, mode='same')


def leggi_csv(percorso):
    if pd is not None:
        df = pd.read_csv(percorso, dtype=np.float32)
    else:
        dati = np.loadtxt(percorso, delimiter=",", skiprows=1, dtype=np.float32)
        return None, dati[:, 1]
    ch1 = df[CANALE_RAMPA].to_numpy() if CANALE_RAMPA and CANALE_RAMPA in df.columns else None
    return ch1, df[CANALE].to_numpy()


def rampa_ch1(ch1):
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 else None


def analizza_campane(ch1, ch2):
    """Tutte le campane di una membrana, sulla curva liscia. Restituisce (lista di campane da sinistra a destra, liscia, (a, b))."""
    y = np.asarray(ch2, dtype=float)
    liscia = moving_average(y, WINDOW_LENGTH)
    mezza = (int(WINDOW_LENGTH) + (int(WINDOW_LENGTH) % 2 == 0)) // 2
    a, b = mezza, len(y) - 1 - mezza
    if ch1 is not None:
        rampa = rampa_ch1(np.asarray(ch1, dtype=float))
        if rampa is None:
            return [], liscia, None
        a, b = max(a, rampa[0]), min(b, rampa[1])
    if b <= a:
        return [], liscia, None
    zona = liscia[a:b + 1]
    segno = 1.0 if CAMPANE_VERSO_ALTO else -1.0
    escursione = zona.max() - zona.min()
    if escursione <= 0:
        return [], liscia, (a, b)
    d = np.diff(y[a:b + 1])
    sigma_grezzo = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2)      # rumore ad alta frequenza di CH2 (robusto ai picchi)
    sigma_liscia = sigma_grezzo / np.sqrt(int(WINDOW_LENGTH))                      # rumore residuo dopo la media mobile
    soglia = max(SOGLIA_PROMINENZA * escursione, SNR_MIN * sigma_liscia)
    picchi, prop = find_peaks(segno * zona, prominence=soglia,
                              distance=int(DISTANZA_MIN if DISTANZA_MIN else WINDOW_LENGTH))
    campane = []
    if len(picchi):
        larg, h_taglio, sx, dx = peak_widths(segno * zona, picchi, rel_height=FRAZIONE_ALTEZZA,
                                             prominence_data=(prop["prominences"], prop["left_bases"], prop["right_bases"]))
        for k, p in enumerate(picchi):
            altezza = float(prop["prominences"][k])
            larghezza = float(dx[k] - sx[k])
            campane.append({"indice": a + int(p), "altezza": altezza, "larghezza": larghezza,
                            "Q": altezza / larghezza, "sx": a + float(sx[k]), "dx": a + float(dx[k]),
                            "livello": float(segno * h_taglio[k]), "valore": float(liscia[a + int(p)])})
    return campane, liscia, (a, b)


def scegli_campana(campane):
    """La campana (o la media) che rappresenta la membrana nella mappa; None se non ce ne sono."""
    if not campane:
        return None
    if CRITERIO_CAMPANA == "piu_prominente":
        return max(campane, key=lambda c: c["altezza"])
    if CRITERIO_CAMPANA == "n":
        return campane[int(N_CAMPANA) - 1] if len(campane) >= int(N_CAMPANA) else None
    if CRITERIO_CAMPANA == "media":
        return {"indice": -1, "altezza": float(np.mean([c["altezza"] for c in campane])),
                "larghezza": float(np.mean([c["larghezza"] for c in campane])),
                "Q": float(np.mean([c["Q"] for c in campane]))}
    raise ValueError("CRITERIO_CAMPANA deve essere 'piu_prominente', 'n' o 'media'")


def indici_reticolo(x, y):
    """Riga e colonna di ogni punto, per un percorso a meandro; None se non e' ricostruibile."""
    if len(x) < 4:
        return None
    dy = np.abs(np.diff(y))
    if dy.max() == 0:
        return None
    righe = np.concatenate([[0], np.cumsum(dy > 0.5 * dy.max())])   # il passaggio alla riga successiva e' un grande salto in Y
    lunghezze = np.bincount(righe)
    if len(set(lunghezze.tolist())) != 1:
        return None
    n_col = int(lunghezze[0])
    colonne = np.empty(len(x), dtype=int)
    for r in range(len(lunghezze)):
        colonne[righe == r] = np.arange(n_col) if r % 2 == 0 else np.arange(n_col)[::-1]
    return righe, colonne

In [ ]:
# Cella 2 - legge il registro dei punti del run scelto
if not CARTELLA_RUN.strip():
    raise ValueError("Scrivi in cella 1 la cartella del run in CARTELLA_RUN, es. '2026-10-06/run_007'")
run_folder = os.path.abspath(CARTELLA_RUN)
registro_path = os.path.join(run_folder, "griglia.csv")
if not os.path.isfile(registro_path):
    raise FileNotFoundError(f"griglia.csv non trovato in {run_folder}: controlla CARTELLA_RUN")
with open(registro_path, newline="") as f:
    registro = list(csv.DictReader(f))
print(f"Run: {run_folder} | {len(registro)} punti")

In [ ]:
# Cella 3 - tutte le campane di ogni membrana (cache in heatmap_q_campane.csv), poi scelta della Q per la mappa
valori_path = os.path.join(run_folder, "heatmap_q_valori.csv")
campane_path = os.path.join(run_folder, "heatmap_q_campane.csv")
posizioni = {int(r["indice"]): (float(r["x_mm"]), float(r["y_mm"])) for r in registro}
trovate = {indice: [] for indice in posizioni}     # indice del punto -> campane, da sinistra a destra

if os.path.isfile(campane_path) and not RICALCOLA:
    with open(campane_path, newline="") as f:
        for r in csv.DictReader(f):
            trovate[int(r["indice_punto"])].append({"indice": int(r["campione"]), "altezza": float(r["altezza_V"]),
                                                   "larghezza": float(r["larghezza_campioni"]), "Q": float(r["Q"])})
    print(f"Riusate le campane salvate in {campane_path} (RICALCOLA = True per rileggere i CSV)")
else:
    t0 = time.time()
    for r in registro:
        indice = int(r["indice"])
        percorso = os.path.join(run_folder, r["file"])
        if not os.path.isfile(percorso):
            print(f"  punto {indice}: file mancante ({r['file']}), salto")
            continue
        ch1, ch2 = leggi_csv(percorso)
        campane, _, _ = analizza_campane(ch1, ch2)
        trovate[indice] = [{"indice": c["indice"], "altezza": c["altezza"], "larghezza": c["larghezza"], "Q": c["Q"]} for c in campane]
        print(f"  punto {indice}: {len(campane)} campane  [{time.time() - t0:.0f} s]")
    with open(campane_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["indice_punto", "n_campana", "campione", "altezza_V", "larghezza_campioni", "Q"])
        for indice, lista in trovate.items():
            for n, c in enumerate(lista, start=1):
                w.writerow([indice, n, c["indice"], c["altezza"], c["larghezza"], c["Q"]])
    print(f"Salvate le campane in {campane_path}")

# quali campane contano: almeno SOGLIA_RUN volte la piu' alta del run (e ALTEZZA_MIN_V se impostata)
altezza_max_run = max((c["altezza"] for lista in trovate.values() for c in lista), default=0.0)
soglia_run = max(SOGLIA_RUN * altezza_max_run, ALTEZZA_MIN_V or 0.0)
print(f"Campana piu' alta del run: {altezza_max_run:.4g} V -> si contano le campane alte almeno {soglia_run:.4g} V")

risultati = []   # una riga per membrana: Q della campana scelta (NaN se non ce n'e')
for indice, (x, y) in posizioni.items():
    valide = [c for c in trovate[indice] if c["altezza"] >= soglia_run]
    scelta = scegli_campana(valide)
    risultati.append({"indice": indice, "x": x, "y": y, "n_campane": len(valide),
                      "Q": scelta["Q"] if scelta else np.nan,
                      "altezza": scelta["altezza"] if scelta else np.nan,
                      "larghezza": scelta["larghezza"] if scelta else np.nan})
with open(valori_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["indice", "x_mm", "y_mm", "n_campane", "Q", "altezza_V", "larghezza_campioni"])
    for r in risultati:
        w.writerow([r["indice"], f"{r['x']:.4f}", f"{r['y']:.4f}", r["n_campane"], r["Q"], r["altezza"], r["larghezza"]])
n_ok = sum(np.isfinite(r["Q"]) for r in risultati)
print(f"{n_ok} membrane su {len(risultati)} con una campana valida. Salvato: {valori_path}")

In [ ]:
# Cella 4 - controllo su una membrana: le campane trovate e la Q che finisce nella mappa
riga = next((r for r in registro if int(r["indice"]) == PUNTO_CONTROLLO), None)
if riga is None:
    raise ValueError(f"Punto {PUNTO_CONTROLLO} non presente nel registro")
ch1, ch2 = leggi_csv(os.path.join(run_folder, riga["file"]))
campane, liscia, zona = analizza_campane(ch1, ch2)
scelta = scegli_campana(campane)

passo = max(1, len(ch2) // 20000)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(np.arange(len(ch2))[::passo], np.asarray(ch2)[::passo], color="gray", lw=0.6, alpha=0.6, label=f"{CANALE} grezzo")
ax.plot(np.arange(len(ch2))[::passo], liscia[::passo], color="red", lw=1.5, label=f"media mobile (N={WINDOW_LENGTH})")
if zona:
    ax.axvspan(zona[0], zona[1], color="orange", alpha=0.12, label="zona di ricerca")
for n, c in enumerate(campane, start=1):
    ax.hlines(c["livello"], c["sx"], c["dx"], color="blue", lw=2)
    ax.plot(c["indice"], c["valore"], "kv", ms=7)
    ax.annotate(f"{n}: Q={c['Q'] / UNITA_Q:.3g}", (c["indice"], c["valore"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9)
ax.set_xlabel("indice del campione")
ax.set_ylabel(CANALE)
ax.legend(loc="best")
titolo = f"Punto {PUNTO_CONTROLLO} (X={riga['x_mm']}, Y={riga['y_mm']}): {len(campane)} campane"
if scelta:
    titolo += f" | nella mappa: Q = {scelta['Q'] / UNITA_Q:.3g} ({CRITERIO_CAMPANA})"
ax.set_title(titolo)
plt.tight_layout()
plt.show()

In [ ]:
# Cella 5 - mappa del quality factor con il valore scritto su ogni membrana
x = np.array([r["x"] for r in risultati])
y = np.array([r["y"] for r in risultati])
q = np.array([r["Q"] for r in risultati]) / UNITA_Q
valido = np.isfinite(q)
if not valido.any():
    raise RuntimeError("Nessuna campana trovata in nessuna membrana: controlla SOGLIA_PROMINENZA, WINDOW_LENGTH e CANALE_RAMPA")
norma = Normalize(vmin=VMIN if VMIN is not None else np.nanmin(q), vmax=VMAX if VMAX is not None else np.nanmax(q))
cmap = plt.get_cmap("viridis")


def colore_testo(valore):
    return "black" if norma(valore) > 0.55 else "white"


modo = MAPPA
reticolo = indici_reticolo(x, y) if modo == "reticolo" else None
if modo == "reticolo" and reticolo is None:
    print("Reticolo non ricostruibile dalle posizioni: uso la mappa 'posizioni'")
    modo = "posizioni"

fig, ax = plt.subplots(figsize=(9, 7))
if modo == "reticolo":
    righe, colonne = reticolo
    Z = np.full((righe.max() + 1, colonne.max() + 1), np.nan)
    Z[righe, colonne] = q
    ax.imshow(np.ma.masked_invalid(Z), cmap=cmap, norm=norma, origin="upper", aspect="equal")
    for k in range(len(q)):
        if valido[k]:
            ax.text(colonne[k], righe[k], FORMATO_NUMERO.format(q[k]), ha="center", va="center", fontsize=9, color=colore_testo(q[k]))
        else:
            ax.text(colonne[k], righe[k], "n.d.", ha="center", va="center", fontsize=8, color="gray")
    ax.set_xticks(range(Z.shape[1]))
    ax.set_xticklabels(range(1, Z.shape[1] + 1))
    ax.set_yticks(range(Z.shape[0]))
    ax.set_yticklabels(range(1, Z.shape[0] + 1))
    ax.set_xlabel("colonna (nel verso della prima riga)")
    ax.set_ylabel("riga (dall'alto)")
else:
    punti = np.column_stack([x, y])
    if len(punti) > 1:
        d = np.hypot(punti[:, None, 0] - punti[None, :, 0], punti[:, None, 1] - punti[None, :, 1])
        passo = np.min(d[d > 0])
    else:
        passo = 0.3
    for k in range(len(q)):
        col = cmap(norma(q[k])) if valido[k] else "lightgray"
        ax.add_patch(Rectangle((x[k] - passo / 2, y[k] - passo / 2), passo, passo, facecolor=col, edgecolor="white", lw=0.8))
        testo = FORMATO_NUMERO.format(q[k]) if valido[k] else "n.d."
        ax.text(x[k], y[k], testo, ha="center", va="center", fontsize=9, color=colore_testo(q[k]) if valido[k] else "gray")
    ax.set_xlim(x.min() - passo, x.max() + passo)
    ax.set_ylim(y.min() - passo, y.max() + passo)
    ax.set_aspect("equal")
    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Y (mm)")
fig.colorbar(ScalarMappable(norm=norma, cmap=cmap), ax=ax, label=f"Q = altezza / larghezza  [{UNITA_Q:g} V per campione]")
ax.set_title(f"Quality factor ({CRITERIO_CAMPANA}) - {os.path.basename(run_folder)}")
plt.tight_layout()
percorso_png = os.path.join(run_folder, "heatmap_quality_factor.png")
fig.savefig(percorso_png, dpi=200)
plt.show()
print(f"Salvata in {percorso_png}")